# ML Pipeline using MLLib


In [0]:
from pyspark.sql.functions import col

# Load dataset
data_path = "/Volumes/barbara_lakehouse/ml_sandbox/data/train.csv"
train_df = spark.read.csv(data_path, header=True, inferSchema=True)

# Cast Boolean columns to int
train_df = train_df.withColumn("PassengerId", col("PassengerId").cast("string")) \
                   .withColumn("VIP", col("VIP").cast("int")) \
                   .withColumn("CryoSleep", col("CryoSleep").cast("int")) \
                   .withColumn("Transported", col("Transported").cast("int")) 

display(train_df)

## Spark Dataframes & MLLib pipeline

In [0]:
# Manual feature engineering using pure PySpark DataFrame operations
# (MLlib Pipeline and RDD operations not available in serverless environment)
 
from pyspark.sql.functions import mean, col, coalesce, lit, when
from pyspark.sql.types import DoubleType
 
# Step A: Define column lists
numerical_cols = ["Age", "RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
categorical_cols = ["HomePlanet", "Destination", "VIP", "CryoSleep"]
label_col = "Transported"  # The target
 
# Step B: Impute numerical columns with mean
print("Step B: Imputing numerical columns...")
means = train_df.select([mean(col(c)).alias(c) for c in numerical_cols]).first().asDict()
 
for num_col in numerical_cols:
    train_df = train_df.withColumn(
        num_col + "_imputed",
        coalesce(col(num_col).cast("double"), lit(means[num_col]))
    )
 
# Step C: String indexing for categorical columns
print("Step C: Indexing categorical columns...")
for cat_col in categorical_cols:
    # Get distinct values using collect (serverless compatible)
    distinct_df = train_df.select(cat_col).distinct()
    distinct_vals = [row[cat_col] for row in distinct_df.collect()]
   
    # Create indexed column using when/otherwise chain
    indexed_col = cat_col + "_indexed"
    result_expr = None
   
    for idx, val in enumerate(distinct_vals):
        if result_expr is None:
            result_expr = when(col(cat_col) == val, lit(float(idx)))
        else:
            result_expr = result_expr.when(col(cat_col) == val, lit(float(idx)))
   
    # Handle nulls/unknowns
    result_expr = result_expr.otherwise(lit(float(len(distinct_vals))))
    train_df = train_df.withColumn(indexed_col, result_expr)
 
# Step D: One-hot encoding for categorical columns
print("Step D: One-hot encoding categorical columns...")
for cat_col in categorical_cols:
    indexed_col = cat_col + "_indexed"
    distinct_df = train_df.select(indexed_col).distinct()
    distinct_vals = sorted([row[indexed_col] for row in distinct_df.collect()])
   
    # Create binary columns for each category (exclude last to avoid multicollinearity)
    for val in distinct_vals[:-1]:
        ohe_col_name = f"{cat_col}_ohe_{int(val)}"
        train_df = train_df.withColumn(
            ohe_col_name,
            when(col(indexed_col) == val, 1.0).otherwise(0.0)
        )
 
print("\nFeature engineering completed!")
print(f"Imputed numerical columns: {[c + '_imputed' for c in numerical_cols]}")
 
# Display sample with available columns
print(f"\nSample of transformed data:")
train_df.select(["Age_imputed", "RoomService_imputed", "HomePlanet_indexed", "Destination_indexed"]).show(5)

In [0]:
# The feature engineering was already completed in Cell 4
# train_df now contains all the transformed columns

# Show a preview of the transformed data
train_df.select(
    ["Age_imputed", "HomePlanet_ohe_1", "HomePlanet_indexed"]
).show(5, truncate=False)

## Decision Tree Classifier 

We extend the pipeline with a decision tree classifier to predict the Transported variable.

In [0]:
%pip install torch
dbutils.library.restartPython()

In [0]:
from pyspark.sql.functions import array, col

# Collect all feature column names from the engineered features
numerical_imputed = [c + "_imputed" for c in ["Age", "RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]]

# Get all one-hot encoded columns (they follow pattern: ColumnName_ohe_N)
ohe_cols = [c for c in train_df.columns if "_ohe_" in c]

print(f"Numerical features: {len(numerical_imputed)}")
print(f"One-hot encoded features: {len(ohe_cols)}")

# Combine all feature columns
feature_cols = numerical_imputed + ohe_cols
print(f"\nTotal features: {len(feature_cols)}")

# Create features as an ARRAY (not Vector) for pyspark.ml.connect
train_df_with_features = train_df.withColumn(
    "features",
    array(*[col(c) for c in feature_cols])
)

# Show sample
print("\nSample with features array:")
train_df_with_features.select("Transported", "features").show(5, truncate=False)

In [0]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import pandas as pd
import numpy as np

# Convert Spark DataFrame to pandas for scikit-learn
print("Converting to pandas DataFrame...")
pandas_df = train_df_with_features.select("Transported", "features").toPandas()

# Extract features array into separate columns
print("Preparing features and labels...")
X = np.array(pandas_df['features'].tolist())
y = pandas_df['Transported'].values

print(f"Dataset shape: X={X.shape}, y={y.shape}")

# Train LogisticRegression using scikit-learn
print("\nTraining LogisticRegression model...")
lr_model = LogisticRegression(
    max_iter=1000,  # Increased to ensure convergence
    C=1.0,  # Inverse of regularization strength
    random_state=42,
    solver='lbfgs'
)

lr_model.fit(X, y)
print("✓ Model trained successfully!")

# Make predictions
print("\nMaking predictions...")
y_pred = lr_model.predict(X)
y_pred_proba = lr_model.predict_proba(X)

# Show sample predictions
print("\nSample predictions:")
results_df = pd.DataFrame({
    'Transported': y[:10],
    'Prediction': y_pred[:10],
    'Probability_0': y_pred_proba[:10, 0],
    'Probability_1': y_pred_proba[:10, 1]
})
print(results_df.to_string(index=False))

In [0]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import pandas as pd

# Calculate metrics
accuracy = accuracy_score(y, y_pred)
f1 = f1_score(y, y_pred)

print(f"Training Accuracy: {accuracy:.4f}")
print(f"Training F1 Score: {f1:.4f}")

# Classification report
print("\nClassification Report:")
print(classification_report(y, y_pred, target_names=['Not Transported', 'Transported']))

# Confusion Matrix
print("\nConfusion Matrix:")
cm = confusion_matrix(y, y_pred)
cm_df = pd.DataFrame(
    cm,
    index=['Actual: Not Transported', 'Actual: Transported'],
    columns=['Predicted: Not Transported', 'Predicted: Transported']
)
print(cm_df)